# Cleaning and Pre-Code

In [ ]:
import json
from datetime import datetime, timedelta
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

data_root = "dataset"

# Cleaning

In [ ]:
# =========================
# 1) Participants base table (participants)
# =========================
p_participants = os.path.join(data_root, "participants.tsv")
participants = pd.read_csv(p_participants, sep="\t")


participants = participants[[
    "participant_id",
    "study_group",
    "recommended_split",
    "age",

]].copy()

participants = participants.rename(columns={
    "recommended_split": "Study Split",
    "age": "Age",
    "study_group": "Study Group"
})

df = participants.copy()

#if missing age use median of column
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Age"] = df["Age"].fillna(df["Age"].median())

#turn study group into one hot encoding
study_group_ohe = pd.get_dummies(
    df["Study Group"],
    prefix="study_group"
)

#one hot encoder
df = df.drop(columns=["Study Group"]).join(study_group_ohe)

# turn recommended split into one hot encoding
split_ohe = pd.get_dummies(
    df["Study Split"],
    prefix="split"
)

df = df.drop(columns=["Study Split"]).join(split_ohe)

master = df.copy()


In [ ]:
# =========================
# 2) Wearable activity (wam)
# =========================
p_wam = os.path.join(data_root, "wearable_activity_monitor", "manifest.tsv")
wam = pd.read_csv(p_wam, sep="\t")

col_hr      = "average_heartrate_bpm"                 
col_oxy     = "average_oxygen_saturation_pct"        
col_pa      = "average_daily_activity"                
col_sleep   = "average_sleep_hours"                  
col_stress  = "average_stress_level"                
col_respire = "average_respiratory_rate_bpm"       


keep_cols = ["participant_id"]
for c in [col_hr, col_oxy, col_pa, col_sleep, col_stress, col_respire]:
    if c in wam.columns:
        keep_cols.append(c)


wam_sub = wam[keep_cols].copy()

rename_map = {}
if col_hr in wam_sub.columns:      rename_map[col_hr] = "Heart Rate Pulse"
if col_oxy in wam_sub.columns:     rename_map[col_oxy] = "Oxygen Saturation"
if col_pa in wam_sub.columns:      rename_map[col_pa] = "Physical Activity"
if col_sleep in wam_sub.columns:   rename_map[col_sleep] = "Sleep"
if col_stress in wam_sub.columns:  rename_map[col_stress] = "Stress"
if col_respire in wam_sub.columns: rename_map[col_respire] = "Respiratory Rate"


wam_sub = wam_sub.rename(columns=rename_map)


for col in ["Heart Rate Pulse", "Oxygen Saturation", "Physical Activity", "Sleep", "Stress"]:
    if col in wam_sub.columns:
        wam_sub[col] = pd.to_numeric(wam_sub[col], errors="coerce")
        wam_sub[col] = wam_sub[col].fillna(wam_sub[col].median())

#if col_respire normalise
if "Respiratory Rate" in wam_sub.columns:
    wam_sub["Respiratory Rate"] = pd.to_numeric(wam_sub["Respiratory Rate"], errors="coerce")
    x = wam_sub["Respiratory Rate"]
    denom = (x.max() - x.min())
    if pd.notna(denom) and denom != 0:
        wam_sub["Respiratory Rate"] = (x - x.min()) / denom

master = master.merge(wam_sub, on="participant_id", how="left")

In [ ]:
# ================================
# 3) Wearable blood glucose (blood glucose)
# =================================
bloodglucose_path = os.path.join(data_root, "wearable_blood_glucose", "manifest.tsv")
bg = pd.read_csv(bloodglucose_path, sep="\t")

blood_glucose = "average_glucose_level_mg_dl" 

keep_cols = ["participant_id"]
if blood_glucose in bg.columns:
    keep_cols.append(blood_glucose)

blood_glucose_p = bg[keep_cols].copy()

# Rename into the name
rename_map = {}
if blood_glucose in blood_glucose_p.columns:
    rename_map[blood_glucose] = "Blood Glucose"
blood_glucose_p = blood_glucose_p.rename(columns=rename_map)


#if missing use media
if "Blood Glucose" in blood_glucose_p.columns:
    blood_glucose_p["Blood Glucose"] = pd.to_numeric(blood_glucose_p["Blood Glucose"], errors="coerce")
    blood_glucose_p["Blood Glucose"] = blood_glucose_p["Blood Glucose"].fillna(
        blood_glucose_p["Blood Glucose"].median()
    )


master = master.merge(blood_glucose_p, on="participant_id", how="left")


In [ ]:
# =========================
# 4) Cardiac ECG (ecg) -
# =========================
ecg_path = os.path.join(data_root, "cardiac_ecg", "manifest.tsv",)
ecg = pd.read_csv(ecg_path, sep="\t")

ecg_pr = "PR" 

keep_cols = ["participant_id"]
if ecg_pr in ecg.columns:
    keep_cols.append(ecg_pr)

ecg_p = ecg[keep_cols].copy()

rename_map = {}
if ecg_pr in ecg_p.columns:
    rename_map[ecg_pr] = "PR Interval"

ecg_p = ecg_p.rename(columns=rename_map)

#if missing use median
if "PR Interval" in ecg_p.columns:
    ecg_p["PR Interval"] = pd.to_numeric(ecg_p["PR Interval"], errors="coerce")
    ecg_p["PR Interval"] = ecg_p["PR Interval"].fillna(ecg_p["PR Interval"].median())

master = master.merge(ecg_p, on="participant_id", how="left")

In [ ]:
# ==========================
# 5) Condition (condition_occurence) -
# =========================
from sklearn.preprocessing import LabelEncoder

condition_path = os.path.join(data_root, "clinical_data", "condition_occurrence.csv")
cond = pd.read_csv(condition_path)

condition_read = "condition_source_value" 
id_col = "person_id"

cond = cond[[id_col, condition_read]].copy()

cond["Condition_clean"] = (
    cond[condition_read]
    .astype(str)
    .str.split(",", n=1)
    .str[-1]
    .str.strip()
    .str.title()
)

cond = cond[cond["Condition_clean"].notna() & (cond["Condition_clean"] != "")]

cond = cond.drop_duplicates(subset=[id_col, "Condition_clean"])

cond_ohe = pd.get_dummies(cond["Condition_clean"], prefix="cond")

cond_multi = pd.concat([cond[[id_col]].reset_index(drop=True), cond_ohe.reset_index(drop=True)], axis=1)

cond_multi = cond_multi.groupby(id_col, as_index=False).max()

new_cols = {}

for c in cond_multi.columns:
    if c.startswith("cond_"):
        new_name = c.replace("cond_", "")
        new_cols[c] = new_name

cond_multi = cond_multi.rename(columns=new_cols)

# Merge into master
master = master.merge(cond_multi, left_on="participant_id", right_on=id_col, how="left").drop(columns=[id_col])

cond_cols = list(new_cols.values())
master[cond_cols] = master[cond_cols].fillna(0).astype(int)


In [ ]:
# =========================
# 6) Measurement
# =========================
p_measurement= os.path.join(data_root, "clinical_data", "measurement.csv")
measurement=pd.read_csv(p_measurement)

person_col="person_id"
type_col="measurement_source_value"
value_col="value_as_number"

measurement=measurement[[person_col, type_col, value_col]].copy()

measurement[value_col]=pd.to_numeric(measurement[value_col],errors="coerce")


targets = {
    "HbA1c": ["hba1c"],          
    "RBC": ["red blood cells", "rbc"],
    "WBC": ["white blood cells", "wbc"],
    "BMI": ["bmi"],
    "Glucose": ["glucose"],
    "Height": ["height"],
    "Cholesterol": ["cholesterol"],
    "Insulin": ["insulin"],
}


# Create Variable column
measurement["Variable"] = pd.NA
src = measurement[type_col].astype(str).str.lower()


for var, keys in targets.items():
    mask = False
    for k in keys:
        mask |= src.str.contains(k, na=False, regex=False)
    measurement.loc[mask, "Variable"] = var

meas = measurement.dropna(subset=["Variable"])

meas_agg = (
    meas
    .groupby([person_col, "Variable"], as_index=False)[value_col]
    .median()
)

meas_wide = (
    meas_agg
    .pivot(index=person_col, columns="Variable", values=value_col)
    .reset_index()
)

for var in targets.keys():
    if var in meas_wide.columns:
        meas_wide[var] = meas_wide[var].fillna(meas_wide[var].median())

master = (
    master
    .merge(meas_wide, left_on="participant_id", right_on=person_col, how="left")
    .drop(columns=[person_col])
)

In [ ]:
# =========================
# Done: master dataframe (one row per participant)
# =========================
master = master.drop_duplicates(subset=["participant_id"])
print("Final shape:", master.shape)
print(master.columns.tolist())
master.head()

In [ ]:
master.to_csv("master_dataframe.csv", index=False)